# Notebook for the experiments conducted in the paper entitled "From Pairwise to Higher-Order Brain Community Detection: A Hypergraph Signal Processing Approach on Brain Functional Connectivity Analysis"

In [19]:
# Auxiliary libraries for the computation and plot of the results
import sys

sys.path.insert(0, "./Background_Scripts")

import pandas as pd
import numpy as np

from Background_Scripts.HCP_Data_Vis_Schaefer_100Parcels import *
from Background_Scripts.plot_functions import *
from Background_Scripts.synthetic_surrogate_functions import (
    generate_phase_randomized_time_series,
    generate_synthetic_time_series,
)
from Background_Scripts.computation_connectivity_weights_functions import *
from Background_Scripts.hypergraph_connectivity_functions import *

# Magic command to load watermark
%load_ext watermark

# Possibility to stop warnings
import warnings

warnings.filterwarnings("ignore")

# Print the versions of used packages in the notebook
%watermark --author "Breno & Fernando" --date --time --python --machine --iversion --watermark


The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Author: Breno & Fernando

Python implementation: CPython
Python version       : 3.9.23
IPython version      : 8.4.0

Compiler    : MSC v.1929 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD
CPU cores   : 32
Architecture: 64bit

permetrics: 2.0.0
scipy     : 1.8.0
tqdm      : 4.67.1
trimesh   : 4.7.0
seaborn   : 0.11.2
sys       : 3.9.23 (main, Jun  5 2025, 13:25:08) [MSC v.1929 64 bit (AMD64)]
networkx  : 2.4
numpy     : 1.22.3
joblib    : 1.5.1
thoi      : 0.2.37
re        : 2.2.1
pandas    : 1.3.5
karateclub: 1.3.3
plotly    : 4.6.0
matplotlib: 3.5.2
sklearn   : 1.6.1
IPython   : 8.4.0

Watermark: 2.5.0



Definition of paths and constants

In [ ]:
# ----------------------
# Directories and files
# ----------------------
time_series_empirical_dir = "./time_series_empirical"
time_series_surrogate_dir = "./time_series_phase_randomized"
time_series_synthetic_hybrid_dir = "./time_series_synthetic_hybrid"
time_series_synthetic_pairwise_dir = "./time_series_synthetic_pairwise"
mi_weights_empirical_dir = "./mi_weights_empirical"
mi_weights_surrogate_dir = "./mi_weights_surrogate"
mi_weights_synthetic_hybrid_dir = "./mi_weights_synthetic_hybrid"
mi_weights_synthetic_pairwise_dir = "./mi_weights_synthetic_pairwise"
hoi_weights_empirical_dir = "./hoi_weights_empirical"
hoi_weights_surrogate_dir = "./hoi_weights_surrogate"
hoi_weights_synthetic_hybrid_dir = "./hoi_weights_synthetic_hybrid"
hoi_weights_synthetic_pairwise_dir = "./hoi_weights_synthetic_pairwise"
hypergraph_modes_empirical_dir = "./hypergraph_modes_empirical"
hypergraph_modes_surrogate_dir = "./hypergraph_modes_surrogate"
hypergraph_modes_synthetic_hybrid_dir = "./hypergraph_modes_synthetic_hybrid"
hypergraph_modes_synthetic_pairwise_dir = "./hypergraph_modes_synthetic_pairwise"
mi_avr_empirical_file = "./mi_avr_empirical.npy"
mi_avr_surrogate_file = "./mi_avr_surrogate.npy"
mi_avr_synthetic_hybrid_file = "./mi_avr_synthetic_hybrid.npy"
mi_avr_synthetic_pairwise_file = "./mi_avr_synthetic_pairwise.npy"
hoi_avr_empirical_file = "./hoi_avr_empirical.npy"
hoi_avr_surrogate_file = "./hoi_avr_surrogate.npy"
hoi_avr_synthetic_hybrid_file = "./hoi_avr_synthetic_hybrid.npy"
hoi_avr_synthetic_pairwise_file = "./hoi_avr_synthetic_pairwise.npy"

# ----------------------------------------
# Hypergraph signal processing parameters
# ----------------------------------------
shift_operator = "laplacian"
norm_type = "sym"
layers = [58]
pre_interactions = 100
interactions = 4000
lamb = 0.1
objectives = {"SI": "max"}

# ------------------------------------------------
# Load subject metadata (e.g., gender) from Excel
# ------------------------------------------------
subject_metadata_file = "./hcp_subject_genders_info.xlsx"
df_subject_metadata = pd.read_excel(
    subject_metadata_file, usecols=["Subject", "Gender"]
)


Computing the pairwise weights for the empirical brain graphs, $\mathcal{G}^{[n]}_{MI} = \{\mathcal{V}, \mathbf{A}^{[n]}_{MI}\}$ for $n\in\{0,1,\cdots,1165\}$, as well as for the mean brain graph, $\mathcal{G}_{MI} = \{\mathcal{V}, \mathbf{A}_{MI}\}$.

In [ ]:
compute_mi_weights_from_time_series(
    input_dir=time_series_empirical_dir,
    output_dir=mi_weights_empirical_dir,
    nbins_mi=20,
    normalized=True,
)
df_mi_empirical = load_connectivity_files(
    mi_weights_empirical_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)

compute_mean_weights(
    df=df_mi_empirical, output_file=mi_avr_empirical_file, zscored=False
)

Computing triple-wise weights of the empirical brain hypergraphs $\mathcal{H}_{OI}^{[n]} = \{\mathcal{V}, \mathcal{M}^{[n]}\}, \mathcal{H}_{TC}^{[n]} = \{\mathcal{V}, \mathcal{T}^{[n]}\}$ for $n\in \{0,1,\cdots,1165\}$ rs-fMRI index and the weights of the mean hypergraphs $\mathcal{H}_{OI} = \{\mathcal{V}, \mathcal{M}\}, \mathcal{H}_{TC} = \{\mathcal{V}, \mathcal{T}\}$. This may take a while.

In [ ]:
compute_oi_tc_weights_from_time_series(
    input_dir=time_series_empirical_dir,
    output_dir=hoi_weights_empirical_dir,
    hoi_labels=all_triangles,
    zscored=False,
)

df_hoi_empirical = load_connectivity_files(
    hoi_weights_empirical_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)

compute_mean_weights(
    df=df_hoi_empirical, output_file=hoi_avr_empirical_file, zscored=True
)

Plot the edge weights histogram of the brain graph $\mathcal{G}_{MI} = \{\mathcal{V}, \mathbf{A}_{MI}\}$ and the hyperedges weights of the brain hypergraphs $\mathcal{H}_{OI} = \{\mathcal{V}, \mathcal{M}\}, \mathcal{H}_{TC} = \{\mathcal{V}, \mathcal{T}\}$. Furthermore, the 3D brain model representations of $\mathcal{G}_{MI}, \mathcal{H}_{OI},\mathcal{H}_{TC}$ are generated.

In [ ]:
plot_weights_histogram(
    mi_avr_empirical_file, hoi_avr_empirical_file, file_name="./weight_distribution.pdf"
)

edge_avr = np.load(mi_avr_empirical_file)
edges = {}
for i, e in enumerate(all_edges):
    edges[e] = edge_avr[i]
hoi_avr = np.load(hoi_avr_empirical_file)
hoi_ii = {}
hoi_tc = {}
for i, triangle in enumerate(all_triangles):
    hoi_ii[triangle] = hoi_avr[i, 0]
    hoi_tc[triangle] = hoi_avr[i, 1]

Plot_Brain_Interactions(
    interaction_weights=edges,
    density=0.05,
    node_feature_range=(15, 75),
    movie="./brain_graph_G.html",
)

Plot_Brain_Interactions(
    interaction_weights=hoi_ii,
    density=0.0015,
    node_feature_range=(15, 75),
    movie="./brain_hypergraph_Hoi.html",
)

Plot_Brain_Interactions(
    interaction_weights=hoi_tc,
    density=0.0015,
    node_feature_range=(15, 75),
    movie="./brain_hypergraph_Htc.html",
)

Generating phase-randomized, hybrid synthetic fMRI-like time series. This may take a while.

In [ ]:
# -------------------------------------------
# Generate the surrogate time series
# -------------------------------------------

generate_phase_randomized_time_series(
    input_dir=time_series_empirical_dir,
    output_dir=time_series_surrogate_dir,
    seed=42,
)

# -------------------------------------------
# Generate the hybrid synthetic time series
# -------------------------------------------

generate_synthetic_time_series(
    input_dir=time_series_empirical_dir,
    triangle_weights_dir=hoi_weights_empirical_dir,
    output_dir=time_series_synthetic_hybrid_dir,
    psi_oi=0.50,
    psi_tc=0.25,
    order=1,
    seed=42,
)


Computing the pairwise weights for the synthetic and surrogate brain graphs, $\widetilde{\mathcal{G}}^{[n]}_{MI} = \{\mathcal{V}, \widetilde{\mathbf{A}}^{[n]}_{MI}\}, \breve{\mathcal{G}}^{[n]}_{MI} = \{\mathcal{V}, \breve{\mathbf{A}}^{[n]}_{MI}\}$ for $n\in\{0,1,\cdots,1165\}$, as well as for the mean brain graph, $\widetilde{\mathcal{G}}_{MI} = \{\mathcal{V}, \widetilde{\mathbf{A}}_{MI}\}, \breve{\mathcal{G}}_{MI} = \{\mathcal{V}, \breve{\mathbf{A}}_{MI}\}$.

In [ ]:
# -------------------------------------------
# Compute MI weights for surrogate time series
# -------------------------------------------

compute_mi_weights_from_time_series(
    input_dir=time_series_surrogate_dir,
    output_dir=mi_weights_surrogate_dir,
    nbins_mi=20,
    normalized=True,
)
df_mi_surrogate = load_connectivity_files(
    mi_weights_surrogate_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)
compute_mean_weights(
    df=df_mi_surrogate, output_file=mi_avr_surrogate_file, zscored=False
)

# -------------------------------------------
# Compute MI weights for synthetic time series
# -------------------------------------------

compute_mi_weights_from_time_series(
    input_dir=time_series_synthetic_hybrid_dir,
    output_dir=mi_weights_synthetic_hybrid_dir,
    nbins_mi=20,
    normalized=True,
)
df_mi_synthetic_hybrid = load_connectivity_files(
    mi_weights_synthetic_hybrid_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)
compute_mean_weights(
    df=df_mi_synthetic_hybrid, output_file=mi_avr_synthetic_hybrid_file, zscored=False
)

Computing triple-wise weights of the surrogate brain hypergraphs $\widetilde{\mathcal{H}}_{OI}^{[n]} = \{\mathcal{V}, \widetilde{\mathcal{M}}^{[n]}\}, \widetilde{\mathcal{H}}_{TC}^{[n]} = \{\mathcal{V}, \widetilde{\mathcal{T}}^{[n]}\}$ and synthetic brain hypergraphs $\breve{\mathcal{H}}_{OI}^{[n]} = \{\mathcal{V}, \breve{\mathcal{M}}^{[n]}\}, \breve{\mathcal{H}}_{TC}^{[n]} = \{\mathcal{V}, \breve{\mathcal{T}}^{[n]}\}$ for $n\in \{0,1,\cdots,1165\}$ rs-fMRI index and the weights of the surrogate mean hypergraphs $\widetilde{\mathcal{H}}_{OI} = \{\mathcal{V}, \widetilde{\mathcal{M}}\}, \widetilde{\mathcal{H}}_{TC} = \{\mathcal{V}, \widetilde{\mathcal{T}}\}$ and the synthetic mean hypergraphs $\breve{\mathcal{H}}_{OI} = \{\mathcal{V}, \breve{\mathcal{M}}\}, \breve{\mathcal{H}}_{TC} = \{\mathcal{V}, \breve{\mathcal{T}}\}$. This may take a while.

In [ ]:
# -------------------------------------------------------
# Compute the HOI weights from the surrogate time series
# -------------------------------------------------------

compute_oi_tc_weights_from_time_series(
    input_dir=time_series_surrogate_dir,
    output_dir=hoi_weights_surrogate_dir,
    hoi_labels=all_triangles,
    zscored=False,
)

df_surrogate_hoi = load_connectivity_files(
    hoi_weights_surrogate_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)

compute_mean_weights(
    df=df_surrogate_hoi, output_file=hoi_avr_surrogate_file, zscored=True
)


# -------------------------------------------------------
# Compute the HOI weights from the synthetic time series
# -------------------------------------------------------

compute_oi_tc_weights_from_time_series(
    input_dir=time_series_synthetic_hybrid_dir,
    output_dir=hoi_weights_synthetic_hybrid_dir,
    hoi_labels=all_triangles,
    zscored=False,
)

df_hybrid_synthetic_hoi = load_connectivity_files(
    hoi_weights_synthetic_hybrid_dir, filename_regex=r"(\d+)_fMRI_REST(\d+)"
)

compute_mean_weights(
    df=df_hybrid_synthetic_hoi, output_file=hoi_avr_synthetic_hybrid_file, zscored=True
)

Plotting the strength of the hypergraph modes derived from the empirical, surrogate and synthetic mean brain hypergraphs

In [ ]:
plot_hypergraph_modes_strength(
    hoi_weights_dir=hoi_avr_empirical_file,
    hoi_weights_surrogate_dir=hoi_avr_surrogate_file,
    hoi_weights_synthetic_dir=hoi_avr_synthetic_hybrid_file,
    hoi_labels=all_triangles,
    n_modes=51,
    file_name="./hypergraph_modes_strengths.pdf",
    label_fontsize=26,
    legend_fontsize=22,
    title_fontsize=30,
    tick_fontsize=24,
    figsize=(10, 9),
)


Computing the clustering models for the mean brain graph $\mathcal{G}_{MI}=\{\mathcal{V}, \mathbf{A}_{MI}\}$ and the selected mean hypergraph modes $\mathcal{G}_{OI}^{(k=0,4)}=\{\mathcal{V}, |\widehat{\mathbf{M}}_s|^{(k=0,4)}\}, \mathcal{G}_{TC}^{(k=0,4)}=\{\mathcal{V}, |\widehat{\mathbf{T}}_s|^{(k=0,4)}\}$, varying the number of partitions $K\in\{2,\cdots,10\}$.

In [ ]:
k_clusters = list(range(2, 11))

df_empirical = []
mi_avr = unpack_upper(np.load(mi_avr_empirical_file), N_rois)
df_empirical.append({"Matrix": mi_avr, "Metric": "mi", "Mode": None})
As_ii, As_tc = get_symmetrized_t_fft(hoi_avr_empirical_file, all_triangles)
Aoi_0_avr = np.abs(As_ii[:, :, 0])
df_empirical.append({"Matrix": Aoi_0_avr, "Metric": "oi", "Mode": 0})
Atc_0_avr = np.abs(As_tc[:, :, 0])
df_empirical.append({"Matrix": Atc_0_avr, "Metric": "tc", "Mode": 0})
Aoi_4_avr = np.abs(As_ii[:, :, 4])
df_empirical.append({"Matrix": Aoi_4_avr, "Metric": "oi", "Mode": 4})
Atc_4_avr = np.abs(As_tc[:, :, 4])
df_empirical.append({"Matrix": Atc_4_avr, "Metric": "tc", "Mode": 4})

df_empirical = pd.DataFrame(df_empirical)
df_empirical["scores_spectral"] = [None] * len(df_empirical)
df_empirical["models_spectral"] = [None] * len(df_empirical)
df_empirical["embeddings_spectral"] = [None] * len(df_empirical)
df_empirical["scores_danmf"] = [None] * len(df_empirical)
df_empirical["models_danmf"] = [None] * len(df_empirical)
df_empirical["embeddings_danmf"] = [None] * len(df_empirical)

for idx, row in df_empirical.iterrows():
    print(
        f"Processing overall original matrix: Metric {row['Metric']}"
        + (
            f", Mode {row['Mode']}"
            if "Mode" in row and not pd.isna(row["Mode"])
            else ""
        )
    )
    results_spectral = graph_spectral_clustering_optimized(
        A=row["Matrix"],
        k_clusters=k_clusters,
        metrics=list(objectives.keys()),
        shift_operator=shift_operator,
        norm_type=norm_type,
        parallel=True,
    )

    results_danmf = danmf_clustering_optimized(
        A=row["Matrix"],
        k_clusters=k_clusters,
        metrics=list(objectives.keys()),
        layers=layers,
        pre_iterations=pre_interactions,
        iterations=interactions,
        lamb=lamb,
        parallel=True,
    )

    df_empirical.at[idx, "scores_spectral"] = results_spectral["scores"]
    df_empirical.at[idx, "models_spectral"] = results_spectral["models"]
    df_empirical.at[idx, "embeddings_spectral"] = results_spectral["embeddings"]
    df_empirical.at[idx, "scores_danmf"] = results_danmf["scores"]
    df_empirical.at[idx, "models_danmf"] = results_danmf["models"]
    df_empirical.at[idx, "embeddings_danmf"] = results_danmf["embeddings"]

Plotting the Silhouette Scores (SCs) and silhouette diagrams across $K$ of the spectral clustering models of $\mathcal{G}_{MI}, \mathcal{G}_{OI}^{(k=0,4)}, \mathcal{G}_{TC}^{(k=0,4)}$.

In [ ]:
_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "mi")][["scores_spectral"]].values[0][0]
    ).T,
    file_name="mi_clustering_metrics_spectral.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{MI}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "mi")]["embeddings_spectral"].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "mi")]["models_spectral"].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{MI}$",
    fig_dir="silhouette_diagrams_mi_spectral.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            ["scores_spectral"]
        ].values[0][0]
    ).T,
    file_name="Aoi_0_clustering_metrics_spectral.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{OI}^{(0)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "embeddings_spectral"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "models_spectral"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{OI}^{(0)}$",
    fig_dir="silhouette_diagrams_Aoi_0_spectral.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            ["scores_spectral"]
        ].values[0][0]
    ).T,
    file_name="Atc_0_clustering_metrics_spectral.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{TC}^{(0)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "embeddings_spectral"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "models_spectral"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{TC}^{(0)}$",
    fig_dir="silhouette_diagrams_Atc_0_spectral.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            ["scores_spectral"]
        ].values[0][0]
    ).T,
    file_name="Aoi_4_clustering_metrics_spectral.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{OI}^{(4)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "embeddings_spectral"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "models_spectral"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{OI}^{(4)}$",
    fig_dir="silhouette_diagrams_Aoi_4_spectral.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            ["scores_spectral"]
        ].values[0][0]
    ).T,
    file_name="Atc_4_clustering_metrics_spectral.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{TC}^{(4)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "embeddings_spectral"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "models_spectral"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{TC}^{(4)}$",
    fig_dir="silhouette_diagrams_Atc_4_spectral.pdf",
    plot_show=True,
)

Select the spectral clustering models that are most consistent with the criteria for good clustering, generating the node clustering distribution and 3D brain communities

In [ ]:
selected_models_spectral = [
    {"Metric": "mi", "Mode": None, "K": 8},
    {"Metric": "oi", "Mode": 0, "K": 7},
    {"Metric": "tc", "Mode": 0, "K": 7},
    {"Metric": "oi", "Mode": 4, "K": 7},
    {"Metric": "tc", "Mode": 4, "K": 5},
]
selected_models_spectral = pd.DataFrame(selected_models_spectral)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "mi")]["models_spectral"].values[0][
            selected_models_spectral[selected_models_spectral["Metric"] == "mi"][
                "K"
            ].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the Spectral clustering model of the brain hypergraph mode $\mathcal{G}_{MI}$",
    file_dir="pie_clusters_mi_spectral.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[(df_empirical["Metric"] == "mi")][
        "models_spectral"
    ].values[0][
        selected_models_spectral[selected_models_spectral["Metric"] == "mi"][
            "K"
        ].values[0]
    ],
    movie="brain_clusters_sc_G.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "models_spectral"
        ].values[0][
            selected_models_spectral[
                (selected_models_spectral["Metric"] == "oi")
                & (selected_models_spectral["Mode"] == 0)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the Spectral clustering model of the brain hypergraph mode $\mathcal{G}_{OI}^{(0)}$",
    file_dir="pie_clusters_Goi_0_spectral.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)
    ]["models_spectral"].values[0][
        selected_models_spectral[
            (selected_models_spectral["Metric"] == "oi")
            & (selected_models_spectral["Mode"] == 0)
        ]["K"].values[0]
    ],
    movie="brain_clusters_sc_Goi_0.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "models_spectral"
        ].values[0][
            selected_models_spectral[
                (selected_models_spectral["Metric"] == "tc")
                & (selected_models_spectral["Mode"] == 0)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the Spectral clustering model of the brain hypergraph mode $\mathcal{G}_{TC}^{(0)}$",
    file_dir="pie_clusters_Gtc_0_spectral.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)
    ]["models_spectral"].values[0][
        selected_models_spectral[
            (selected_models_spectral["Metric"] == "tc")
            & (selected_models_spectral["Mode"] == 0)
        ]["K"].values[0]
    ],
    movie="brain_clusters_sc_Gtc_0.html",
    plot_show=False,
)


plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "models_spectral"
        ].values[0][
            selected_models_spectral[
                (selected_models_spectral["Metric"] == "oi")
                & (selected_models_spectral["Mode"] == 4)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the Spectral clustering model of the brain hypergraph mode $\mathcal{G}_{OI}^{(4)}$",
    file_dir="pie_clusters_Goi_4_spectral.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)
    ]["models_spectral"].values[0][
        selected_models_spectral[
            (selected_models_spectral["Metric"] == "oi")
            & (selected_models_spectral["Mode"] == 4)
        ]["K"].values[0]
    ],
    movie="brain_clusters_sc_Goi_4.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "models_spectral"
        ].values[0][
            selected_models_spectral[
                (selected_models_spectral["Metric"] == "tc")
                & (selected_models_spectral["Mode"] == 4)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the Spectral clustering model of the brain hypergraph mode $\mathcal{G}_{TC}^{(4)}$",
    file_dir="pie_clusters_Gtc_4_spectral.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)
    ]["models_spectral"].values[0][
        selected_models_spectral[
            (selected_models_spectral["Metric"] == "tc")
            & (selected_models_spectral["Mode"] == 4)
        ]["K"].values[0]
    ],
    movie="brain_clusters_sc_Gtc_4.html",
    plot_show=False,
)

Plotting the Silhouette Scores (SCs) and sihouette diagrams across $K$ of the DANMF-based clustering models of $\mathcal{G}_{MI}, \mathcal{G}_{OI}^{(k=0,4)}, \mathcal{G}_{TC}^{(k=0,4)}$.

In [ ]:
_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "mi")][["scores_danmf"]].values[0][0]
    ).T,
    file_name="mi_clustering_metrics_danmf.pdf",
    title="Validation metrics vs number of clusters (k): $\mathcal{G}_{MI}$",
    show=True,
    objectives=objectives,
    normalize=False,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "mi")]["embeddings_danmf"].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "mi")]["models_danmf"].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{MI}$",
    fig_dir="silhouette_diagrams_mi_danmf.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            ["scores_danmf"]
        ].values[0][0]
    ).T,
    file_name="Aoi_0_clustering_metrics_danmf.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{OI}^{(0)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "embeddings_danmf"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "models_danmf"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{OI}^{(0)}$",
    fig_dir="silhouette_diagrams_Aoi_0_danmf.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            ["scores_danmf"]
        ].values[0][0]
    ).T,
    file_name="Atc_0_clustering_metrics_danmf.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{TC}^{(0)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "embeddings_danmf"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "models_danmf"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{TC}^{(0)}$",
    fig_dir="silhouette_diagrams_Atc_0_danmf.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            ["scores_danmf"]
        ].values[0][0]
    ).T,
    file_name="Aoi_4_clustering_metrics_danmf.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{OI}^{(4)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "embeddings_danmf"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "models_danmf"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{OI}^{(4)}$",
    fig_dir="silhouette_diagrams_Aoi_4_danmf.pdf",
    plot_show=True,
)

_ = plot_clustering_performance_metrics(
    df_plot=pd.DataFrame(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            ["scores_danmf"]
        ].values[0][0]
    ).T,
    file_name="Atc_4_clustering_metrics_danmf.pdf",
    title="Validation metrics vs number of clusters ($K$): $\mathcal{G}_{TC}^{(4)}$",
    show=True,
    objectives=objectives,
)

plot_models_silhouette_diagrams(
    list_kclusters=k_clusters,
    list_embeddings=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "embeddings_danmf"
        ].values[0]
    ),
    list_kmeans_models=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "models_danmf"
        ].values[0]
    ),
    nodes_color=schaefer_nodes_color,
    name_dataset=r"$\mathcal{G}_{TC}^{(4)}$",
    fig_dir="silhouette_diagrams_Atc_4_danmf.pdf",
    plot_show=True,
)

Select the DANMF-based clustering models that are most consistent with the criteria for good clustering, generating the node clustering distribution and 3D brain communities

In [ ]:
selected_models_danmf = [
    {"Metric": "mi", "Mode": None, "K": 8},
    {"Metric": "oi", "Mode": 0, "K": 9},
    {"Metric": "tc", "Mode": 0, "K": 9},
    {"Metric": "oi", "Mode": 4, "K": 7},
    {"Metric": "tc", "Mode": 4, "K": 5},
]
selected_models_danmf = pd.DataFrame(selected_models_danmf)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "mi")]["models_danmf"].values[0][
            selected_models_danmf[selected_models_danmf["Metric"] == "mi"]["K"].values[
                0
            ]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the DANMF-based clustering model of the brain hypergraph mode $\mathcal{G}_{MI}$",
    file_dir="pie_clusters_mi_danmf.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[(df_empirical["Metric"] == "mi")]["models_danmf"].values[
        0
    ][selected_models_danmf[selected_models_danmf["Metric"] == "mi"]["K"].values[0]],
    movie="brain_clusters_danmf_G.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)][
            "models_danmf"
        ].values[0][
            selected_models_danmf[
                (selected_models_danmf["Metric"] == "oi")
                & (selected_models_danmf["Mode"] == 0)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the DANMF-based clustering model of the brain hypergraph mode $\mathcal{G}_{OI}^{(0)}$",
    file_dir="pie_clusters_Goi_0_danmf.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 0)
    ]["models_danmf"].values[0][
        selected_models_danmf[
            (selected_models_danmf["Metric"] == "oi")
            & (selected_models_danmf["Mode"] == 0)
        ]["K"].values[0]
    ],
    movie="brain_clusters_danmf_Goi_0.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)][
            "models_danmf"
        ].values[0][
            selected_models_danmf[
                (selected_models_danmf["Metric"] == "tc")
                & (selected_models_danmf["Mode"] == 0)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the DANMF-based clustering model of the brain hypergraph mode $\mathcal{G}_{TC}^{(0)}$",
    file_dir="pie_clusters_Gtc_0_danmf.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 0)
    ]["models_danmf"].values[0][
        selected_models_danmf[
            (selected_models_danmf["Metric"] == "tc")
            & (selected_models_danmf["Mode"] == 0)
        ]["K"].values[0]
    ],
    movie="brain_clusters_danmf_Gtc_0.html",
    plot_show=False,
)


plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)][
            "models_danmf"
        ].values[0][
            selected_models_danmf[
                (selected_models_danmf["Metric"] == "oi")
                & (selected_models_danmf["Mode"] == 4)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the DANMF-based clustering model of the brain hypergraph mode $\mathcal{G}_{OI}^{(4)}$",
    file_dir="pie_clusters_Goi_4_danmf.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "oi") & (df_empirical["Mode"] == 4)
    ]["models_danmf"].values[0][
        selected_models_danmf[
            (selected_models_danmf["Metric"] == "oi")
            & (selected_models_danmf["Mode"] == 4)
        ]["K"].values[0]
    ],
    movie="brain_clusters_danmf_Goi_4.html",
    plot_show=False,
)

plot_node_clustering_distribution(
    nodes_cluster=(
        df_empirical[(df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)][
            "models_danmf"
        ].values[0][
            selected_models_danmf[
                (selected_models_danmf["Metric"] == "tc")
                & (selected_models_danmf["Mode"] == 4)
            ]["K"].values[0]
        ]
    ),
    nodes_color=schaefer_nodes_color,
    nodes_subnet=schaefer_nodes_subnet,
    fig_title=r"Nodes distribution of the DANMF-based clustering model of the brain hypergraph mode $\mathcal{G}_{TC}^{(4)}$",
    file_dir="pie_clusters_Gtc_4_danmf.pdf",
    plot_show=True,
    ncols=4,
    hole_width=0.6,
    label_fontsize=20,
    pct_fontsize=50,
    pct2_fontsize=50,
    legend_fontsize=40,
    center_fontsize=60,
    center_fontweight="bold",
    center_color="black",
    figsize_per_col=5,
    figsize_per_row=5,
)

Plot_Brain_Clusters(
    nodes_cluster=df_empirical[
        (df_empirical["Metric"] == "tc") & (df_empirical["Mode"] == 4)
    ]["models_danmf"].values[0][
        selected_models_danmf[
            (selected_models_danmf["Metric"] == "tc")
            & (selected_models_danmf["Mode"] == 4)
        ]["K"].values[0]
    ],
    movie="brain_clusters_danmf_Gtc_4.html",
    plot_show=False,
)


Computing the hypergraph modes ${\mathcal{G}_{OI}^{(k=0,4)}}^{[n]}, {\mathcal{G}_{TC}^{(k=0,4)}}^{[n]}$ at the individual-level from the empirical, hybrid-synthetic and surrogaye data

In [ ]:
# --------------------------------------------
# Compute hypergraph modes for empirical data
# --------------------------------------------
compute_hypergraph_modes(
    hoi_dir=hoi_weights_empirical_dir,
    output_dir=hypergraph_modes_empirical_dir,
    hoi_labels=all_triangles,
    modes=[0, 4],
    absolute=True,
)

df_hypergraph_modes = load_connectivity_files(
    hypergraph_modes_empirical_dir,
    filename_regex=r"(\d+)_fMRI_REST(\d+)_A_(oi|tc)_(\d+)",
)

df_Aoi_0_empirical = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Aoi_4_empirical = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

df_Atc_0_empirical = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Atc_4_empirical = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

# --------------------------------------------
# Compute hypergraph modes for surrogate data
# --------------------------------------------
compute_hypergraph_modes(
    hoi_dir=hoi_weights_surrogate_dir,
    output_dir=hypergraph_modes_surrogate_dir,
    hoi_labels=all_triangles,
    modes=[0, 4],
    absolute=True,
)

df_hypergraph_modes = load_connectivity_files(
    hypergraph_modes_surrogate_dir,
    filename_regex=r"(\d+)_fMRI_REST(\d+)_A_(oi|tc)_(\d+)",
)

df_Aoi_0_surrogate = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Aoi_4_surrogate = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

df_Atc_0_surrogate = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Atc_4_surrogate = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

# --------------------------------------------
# Compute hypergraph modes for synthetic data
# --------------------------------------------
compute_hypergraph_modes(
    hoi_dir=hoi_weights_synthetic_hybrid_dir,
    output_dir=hypergraph_modes_synthetic_hybrid_dir,
    hoi_labels=all_triangles,
    modes=[0, 4],
    absolute=True,
)

df_hypergraph_modes = load_connectivity_files(
    hypergraph_modes_synthetic_hybrid_dir,
    filename_regex=r"(\d+)_fMRI_REST(\d+)_A_(oi|tc)_(\d+)",
)

df_Aoi_0_synthetic_hybrid = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Aoi_4_synthetic_hybrid = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "oi"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

df_Atc_0_synthetic_hybrid = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 0
].reset_index(drop=True)

df_Atc_4_synthetic_hybrid = df_hypergraph_modes[df_hypergraph_modes["Metric"] == "tc"][
    df_hypergraph_modes["Mode"] == 4
].reset_index(drop=True)

Test-retest community consistency analysis of the $\mathcal{G}_{MI}^{[n]}$ and its surrogate and hybrid-synthetic versions using spectral clustering algorithm

In [ ]:
k_clusters = list(range(2, 11))

df_mi_empirical_nmis_subject = consistent_communities(
    df_rest=df_mi_empirical,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_mi_empirical_nmis_subject.to_pickle("df_mi_empirical_nmis_subject_spectral.pkl")

df_mi_surrogate_nmis_subject = consistent_communities(
    df_rest=df_mi_surrogate,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_mi_surrogate_nmis_subject.to_pickle("df_mi_surrogate_nmis_subject_spectral.pkl")

df_mi_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_mi_synthetic_hybrid,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_mi_synthetic_hybrid_nmis_subject.to_pickle(
    "df_mi_synthetic_hybrid_nmis_subject_spectral.pkl"
)


plot_nmi_curves(
    df_list=[
        df_mi_empirical_nmis_subject,
        df_mi_synthetic_hybrid_nmis_subject,
        df_mi_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{MI}$",
    show=True,
    save_path="./mi_nmi_curves_spectral.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the $\mathcal{G}_{MI}^{[n]}$ and its surrogate and hybrid-synthetic versions using DANMF-based clustering algorithm

In [ ]:
df_mi_empirical_nmis_subject = consistent_communities(
    df_rest=df_mi_empirical,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_mi_empirical_nmis_subject.to_pickle("df_mi_empirical_nmis_subject_danmf.pkl")

df_mi_surrogate_nmis_subject = consistent_communities(
    df_rest=df_mi_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_mi_surrogate_nmis_subject.to_pickle("df_mi_surrogate_nmis_subject_danmf.pkl")

df_mi_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_mi_synthetic_hybrid,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_mi_synthetic_hybrid_nmis_subject.to_pickle(
    "df_mi_synthetic_hybrid_nmis_subject_danmf.pkl"
)

plot_nmi_curves(
    df_list=[
        df_mi_empirical_nmis_subject,
        df_mi_synthetic_hybrid_nmis_subject,
        df_mi_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{MI}$",
    show=True,
    save_path="./mi_nmi_curves_danmf.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{OI}^{(0)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using spectral clustering algorithm

In [ ]:
df_Aoi_0_empirical_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_empirical,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_0_empirical_nmis_subject.to_pickle(
    "df_Aoi_0_empirical_nmis_subject_spectral.pkl"
)

df_Aoi_0_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_surrogate,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_0_surrogate_nmis_subject.to_pickle(
    "df_Aoi_0_surrogate_nmis_subject_spectral.pkl"
)

df_Aoi_0_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_synthetic_hybrid,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_0_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Aoi_0_synthetic_hybrid_nmis_subject_spectral.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Aoi_0_empirical_nmis_subject,
        df_Aoi_0_synthetic_hybrid_nmis_subject,
        df_Aoi_0_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{OI}^{(0)}$",
    show=True,
    save_path="./Aoi_0_nmi_curves_spectral.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{OI}^{(0)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using DANMF-based clustering algorithm

In [ ]:
df_Aoi_0_empirical_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_empirical,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_0_empirical_nmis_subject.to_pickle("df_Aoi_0_empirical_nmis_subject_danmf.pkl")

df_Aoi_0_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_0_surrogate_nmis_subject.to_pickle("df_Aoi_0_surrogate_nmis_subject_danmf.pkl")

df_Aoi_0_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Aoi_0_synthetic_hybrid,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_0_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Aoi_0_synthetic_hybrid_nmis_subject_danmf.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Aoi_0_empirical_nmis_subject,
        df_Aoi_0_synthetic_hybrid_nmis_subject,
        df_Aoi_0_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{OI}^{(0)}$",
    show=True,
    save_path="./Aoi_0_nmi_curves_danmf.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{TC}^{(0)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using spectral clustering algorithm

In [ ]:
df_Atc_0_empirical_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_empirical,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_0_empirical_nmis_subject.to_pickle(
    "df_Atc_0_empirical_nmis_subject_spectral.pkl"
)

df_Atc_0_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_surrogate,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_0_surrogate_nmis_subject.to_pickle(
    "df_Atc_0_surrogate_nmis_subject_spectral.pkl"
)

df_Atc_0_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_synthetic_hybrid,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_0_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Atc_0_synthetic_hybrid_nmis_subject_spectral.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Atc_0_empirical_nmis_subject,
        df_Atc_0_synthetic_hybrid_nmis_subject,
        df_Atc_0_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{TC}^{(0)}$",
    show=True,
    save_path="./Atc_0_nmi_curves_spectral.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{TC}^{(0)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using DANMF-based clustering algorithm

In [ ]:
df_Atc_0_empirical_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_empirical,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_0_empirical_nmis_subject.to_pickle("df_Atc_0_empirical_nmis_subject_danmf.pkl")

df_Atc_0_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_0_surrogate_nmis_subject.to_pickle("df_Atc_0_surrogate_nmis_subject_danmf.pkl")

df_Atc_0_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Atc_0_synthetic_hybrid,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_0_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Atc_0_synthetic_hybrid_nmis_subject_danmf.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Atc_0_empirical_nmis_subject,
        df_Atc_0_synthetic_hybrid_nmis_subject,
        df_Atc_0_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{TC}^{(0)}$",
    show=True,
    save_path="./Atc_0_nmi_curves_danmf.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{OI}^{(4)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using spectral clustering algorithm

In [ ]:
df_Aoi_4_empirical_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_empirical,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_4_empirical_nmis_subject.to_pickle(
    "df_Aoi_4_empirical_nmis_subject_spectral.pkl"
)

df_Aoi_4_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_surrogate,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_4_surrogate_nmis_subject.to_pickle(
    "df_Aoi_4_surrogate_nmis_subject_spectral.pkl"
)

df_Aoi_4_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_synthetic_hybrid,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Aoi_4_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Aoi_4_synthetic_hybrid_nmis_subject_spectral.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Aoi_4_empirical_nmis_subject,
        df_Aoi_4_synthetic_hybrid_nmis_subject,
        df_Aoi_4_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{OI}^{(4)}$",
    show=True,
    save_path="./Aoi_4_nmi_curves_spectral.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{OI}^{(4)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using DANMF-based clustering algorithm

In [ ]:
df_Aoi_4_empirical_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_empirical,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_4_empirical_nmis_subject.to_pickle("df_Aoi_4_empirical_nmis_subject_danmf.pkl")

df_Aoi_4_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_4_surrogate_nmis_subject.to_pickle("df_Aoi_4_surrogate_nmis_subject_danmf.pkl")

df_Aoi_4_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Aoi_4_synthetic_hybrid,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Aoi_4_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Aoi_4_synthetic_hybrid_nmis_subject_danmf.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Aoi_4_empirical_nmis_subject,
        df_Aoi_4_synthetic_hybrid_nmis_subject,
        df_Aoi_4_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{OI}^{(4)}$",
    show=True,
    save_path="./Aoi_4_nmi_curves_danmf.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{TC}^{(4)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using spectral clustering algorithm

In [ ]:
df_Atc_4_empirical_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_empirical,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_4_empirical_nmis_subject.to_pickle(
    "df_Atc_4_empirical_nmis_subject_spectral.pkl"
)

df_Atc_4_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_surrogate,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_4_surrogate_nmis_subject.to_pickle(
    "df_Atc_4_surrogate_nmis_subject_spectral.pkl"
)

df_Atc_4_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_synthetic_hybrid,
    k_clusters=k_clusters,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
    danmf=False,
)

df_Atc_4_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Atc_4_synthetic_hybrid_nmis_subject_spectral.pkl"
)

plot_nmi_curves(
    df_list=[
        df_Atc_4_empirical_nmis_subject,
        df_Atc_4_synthetic_hybrid_nmis_subject,
        df_Atc_4_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{TC}^{(4)}$",
    show=True,
    save_path="./Atc_4_nmi_curves_spectral.pdf",
    colors=["C0", "C2", "C1"],
)


Test-retest community consistency analysis of the ${\mathcal{G}_{TC}^{(4)}}^{[n]}$ and its surrogate and hybrid-synthetic versions using DANMF-based clustering algorithm

In [ ]:
df_Atc_4_empirical_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_empirical,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_4_empirical_nmis_subject.to_pickle("df_Atc_4_empirical_nmis_subject_danmf.pkl")

df_Atc_4_surrogate_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_4_surrogate_nmis_subject.to_pickle("df_Atc_4_surrogate_nmis_subject_danmf.pkl")

df_Atc_4_synthetic_hybrid_nmis_subject = consistent_communities(
    df_rest=df_Atc_4_synthetic_hybrid,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)

df_Atc_4_synthetic_hybrid_nmis_subject.to_pickle(
    "df_Atc_4_synthetic_hybrid_nmis_subject_danmf.pkl"
)


plot_nmi_curves(
    df_list=[
        df_Atc_4_empirical_nmis_subject,
        df_Atc_4_synthetic_hybrid_nmis_subject,
        df_Atc_4_surrogate_nmis_subject,
    ],
    labels=["Empirical", "Synthetic", "Surrogate"],
    title=r"Test–retest consistency of individual brain communities: $\mathcal{G}_{TC}^{(4)}$",
    show=True,
    save_path="./Atc_4_nmi_curves_danmf.pdf",
    colors=["C0", "C2", "C1"],
)


Generation of the pairwise-only synthetic time series, computation of its HOI weights and the hypergraph modes at individual-level

In [ ]:
# -------------------------------------------
# Generate the hybrid synthetic time series
# -------------------------------------------

generate_synthetic_time_series(
    input_dir=time_series_empirical_dir,
    triangle_weights_dir=hoi_weights_empirical_dir,
    output_dir=time_series_synthetic_pairwise_dir,
    psi_ii=0.0,
    psi_tc=0.0,
    order=1,
    seed=42,
)

# -------------------------------------------------------
# Compute the HOI weights from the synthetic time series
# -------------------------------------------------------

compute_oi_tc_weights_from_time_series(
    input_dir=time_series_synthetic_pairwise_dir,
    output_dir=hoi_weights_synthetic_pairwise_dir,
    hoi_labels=all_triangles,
    zscored=False,
)

# --------------------------------------------
# Compute hypergraph modes for pairwise-only synthetic data
# --------------------------------------------
compute_hypergraph_modes(
    hoi_dir=hoi_weights_synthetic_pairwise_dir,
    output_dir=hypergraph_modes_synthetic_pairwise_dir,
    hoi_labels=all_triangles,
    modes=[0, 4],
    absolute=True,
)

df_hypergraph_modes = load_connectivity_files(
    hypergraph_modes_synthetic_pairwise_dir,
    filename_regex=r"(\d+)_fMRI_REST(\d+)_A_(oi|tc)_(\d+)",
)

df_Aoi_0_synthetic_pairwise = df_hypergraph_modes[
    df_hypergraph_modes["Metric"] == "oi"
][df_hypergraph_modes["Mode"] == 0].reset_index(drop=True)

df_Aoi_4_synthetic_pairwise = df_hypergraph_modes[
    df_hypergraph_modes["Metric"] == "oi"
][df_hypergraph_modes["Mode"] == 4].reset_index(drop=True)

df_Atc_0_synthetic_pairwise = df_hypergraph_modes[
    df_hypergraph_modes["Metric"] == "tc"
][df_hypergraph_modes["Mode"] == 0].reset_index(drop=True)

df_Atc_4_synthetic_pairwise = df_hypergraph_modes[
    df_hypergraph_modes["Metric"] == "tc"
][df_hypergraph_modes["Mode"] == 4].reset_index(drop=True)


Computing the community similarity across empirical, (hybrid and pairwise-only) synthetic, and surrogate models of the hypergraph modes using the spectral clustering algorithm

In [ ]:
df_nmis_emp_syn_sur_Aoi_0 = simmilarity_emp_syn_sur(
    df_emp=df_Aoi_0_empirical,
    df_syn_hoi=df_Aoi_0_synthetic_hybrid,
    df_syn_pair=df_Aoi_0_synthetic_pairwise,
    df_sur=df_Aoi_0_surrogate,
    k_clusters=k_clusters,
    danmf=False,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
)
df_nmis_emp_syn_sur_Aoi_0.to_pickle("df_nmis_emp_syn_sur_Aoi_0_spectral.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Aoi_0,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{OI}^{(0)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Aoi_0_spectral.pdf",
)

df_nmis_emp_syn_sur_Atc_0 = simmilarity_emp_syn_sur(
    df_emp=df_Atc_0_empirical,
    df_syn_hoi=df_Atc_0_synthetic_hybrid,
    df_syn_pair=df_Atc_0_synthetic_pairwise,
    df_sur=df_Atc_0_surrogate,
    k_clusters=k_clusters,
    danmf=False,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
)
df_nmis_emp_syn_sur_Atc_0.to_pickle("df_nmis_emp_syn_sur_Atc_0_spectral.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Atc_0,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{TC}^{(0)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Atc_0_spectral.pdf",
)

df_nmis_emp_syn_sur_Aoi_4 = simmilarity_emp_syn_sur(
    df_emp=df_Aoi_4_empirical,
    df_syn_hoi=df_Aoi_4_synthetic_hybrid,
    df_syn_pair=df_Aoi_4_synthetic_pairwise,
    df_sur=df_Aoi_4_surrogate,
    k_clusters=k_clusters,
    danmf=False,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
)
df_nmis_emp_syn_sur_Aoi_4.to_pickle("df_nmis_emp_syn_sur_Aoi_4_spectral.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Aoi_4,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{OI}^{(4)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Aoi_4_spectral.pdf",
)

df_nmis_emp_syn_sur_Atc_4 = simmilarity_emp_syn_sur(
    df_emp=df_Atc_4_empirical,
    df_syn_hoi=df_Atc_4_synthetic_hybrid,
    df_syn_pair=df_Atc_4_synthetic_pairwise,
    df_sur=df_Atc_4_surrogate,
    k_clusters=k_clusters,
    danmf=False,
    spectral_shift_operator=shift_operator,
    spectral_norm_type=norm_type,
)
df_nmis_emp_syn_sur_Atc_4.to_pickle("df_nmis_emp_syn_sur_Atc_4_spectral.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Atc_4,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{TC}^{(4)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Atc_4_spectral.pdf",
)


Computing the community similarity across empirical, (hybrid and pairwise-only) synthetic, and surrogate models of the hypergraph modes using the DANMF-based clustering algorithm

In [ ]:
df_nmis_emp_syn_sur_Aoi_0 = simmilarity_emp_syn_sur(
    df_emp=df_Aoi_0_empirical,
    df_syn_hoi=df_Aoi_0_synthetic_hybrid,
    df_syn_pair=df_Aoi_0_synthetic_pairwise,
    df_sur=df_Aoi_0_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)
df_nmis_emp_syn_sur_Aoi_0.to_pickle("df_nmis_emp_syn_sur_Aoi_0_danmf.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Aoi_0,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{OI}^{(0)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Aoi_0_danmf.pdf",
)

df_nmis_emp_syn_sur_Atc_0 = simmilarity_emp_syn_sur(
    df_emp=df_Atc_0_empirical,
    df_syn_hoi=df_Atc_0_synthetic_hybrid,
    df_syn_pair=df_Atc_0_synthetic_pairwise,
    df_sur=df_Atc_0_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)
df_nmis_emp_syn_sur_Atc_0.to_pickle("df_nmis_emp_syn_sur_Atc_0_danmf.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Atc_0,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{TC}^{(0)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Atc_0_danmf.pdf",
)

df_nmis_emp_syn_sur_Aoi_4 = simmilarity_emp_syn_sur(
    df_emp=df_Aoi_4_empirical,
    df_syn_hoi=df_Aoi_4_synthetic_hybrid,
    df_syn_pair=df_Aoi_4_synthetic_pairwise,
    df_sur=df_Aoi_4_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)
df_nmis_emp_syn_sur_Aoi_4.to_pickle("df_nmis_emp_syn_sur_Aoi_4_danmf.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Aoi_4,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{OI}^{(4)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Aoi_4_danmf.pdf",
)

df_nmis_emp_syn_sur_Atc_4 = simmilarity_emp_syn_sur(
    df_emp=df_Atc_4_empirical,
    df_syn_hoi=df_Atc_4_synthetic_hybrid,
    df_syn_pair=df_Atc_4_synthetic_pairwise,
    df_sur=df_Atc_4_surrogate,
    k_clusters=k_clusters,
    danmf=True,
    danmf_layers=layers,
    danmf_pre_iterations=pre_interactions,
    danmf_iterations=interactions,
    danmf_lamb=lamb,
)
df_nmis_emp_syn_sur_Atc_4.to_pickle("df_nmis_emp_syn_sur_Atc_4_danmf.pkl")

plot_nmis_emp_syn_sur(
    df_nmis=df_nmis_emp_syn_sur_Atc_4,
    k_clusters=k_clusters,
    fig_size=(8, 5),
    fig_title=r"$\mathcal{G}_{TC}^{(4)}$",
    label_fontsize=18,
    tick_fontsize=18,
    title_fontsize=18,
    legend_fontsize=14,
    text_fontsize=19,
    markersize=20,
    fig_name="nmi_emp_syn_sur_Atc_4_danmf.pdf",
)


Load the raw values of the volunteers' HOI hyperedge weights, and compute the strength of the brain hypergraph modes ${\mathcal{G}^{(k=0,4)}_{II_{1}}}^{[i]} = \left\{\mathcal{V}, \left|\widehat{\mathbf{M}}_{s_{1}}^{(k=0,4)}\right|^{[i]}\right\}$, ${\mathcal{G}^{(k=0,4)}_{TC_{1}}}^{[i]} = \left\{\mathcal{V}, \left|\widehat{\mathbf{T}}_{s_{1}}^{(k=0,4)}\right|^{[i]}\right\}$ from the first fMRI recording (REST1), and ${\mathcal{G}^{(k=0,4)}_{II_{2}}}^{[i]} = \left\{\mathcal{V}, \left|\widehat{\mathbf{M}}_{s_{2}}^{(k=0,4)}\right|^{[i]}\right\}$, ${\mathcal{G}^{(k=0,4)}_{TC_{2}}}^{[i]} = \left\{\mathcal{V}, \left|\widehat{\mathbf{T}}_{s_{2}}^{(k=0,4)}\right|^{[i]}\right\}$ from the second fMRI recording (REST2) of the $i$-th individual.

In [ ]:
strength_Aoi_0 = []
strength_Aoi_4 = []
strength_Atc_0 = []
strength_Atc_4 = []
for idx in range(len(df_Aoi_0_empirical)):
    strength_Aoi_0.append(np.mean(np.load(df_Aoi_0_empirical["File"].values[idx])))
    strength_Aoi_4.append(np.mean(np.load(df_Aoi_4_empirical["File"].values[idx])))
    strength_Atc_0.append(np.mean(np.load(df_Atc_0_empirical["File"].values[idx])))
    strength_Atc_4.append(np.mean(np.load(df_Atc_4_empirical["File"].values[idx])))
df_strengths = pd.DataFrame(
    {
        "Subject": df_Aoi_0_empirical["Subject"],
        "REST": df_Aoi_0_empirical["REST"],
        "Strength_Aoi_0": strength_Aoi_0,
        "Strength_Aoi_4": strength_Aoi_4,
        "Strength_Atc_0": strength_Atc_0,
        "Strength_Atc_4": strength_Atc_4,
    }
)


Plotting the strength correlations between the test and retest rs-fMRI scans of the brain hypergraph modes ${\mathcal{G}^{(k=0,4)}_{II_{1,2}}}^{[n]}$, ${\mathcal{G}^{(k=0,4)}_{TC_{1,2}}}^{[n]}$ at the individual level.

In [ ]:
list_metrics = "Strength_Aoi_0"
list_xlabels = "Connection strength of ${\mathcal{G}^{(0)}_{OI_{1}}}^{[i]}$"
list_ylabels = "Connection strength of ${\mathcal{G}^{(0)}_{OI_{2}}}^{[i]}$"
plot_correlation_scatters(
    df_strengths,
    list_metrics,
    list_xlabels,
    list_ylabels,
    figsize=(8, 4),
    fig_dir="./correlation_strength_values_Aoi_0.pdf",
)

list_metrics = "Strength_Aoi_4"
list_xlabels = "Connection strength of ${\mathcal{G}^{(4)}_{OI_{1}}}^{[i]}$"
list_ylabels = "Connection strength of ${\mathcal{G}^{(4)}_{OI_{2}}}^{[i]}$"
plot_correlation_scatters(
    df_strengths,
    list_metrics,
    list_xlabels,
    list_ylabels,
    figsize=(8, 4),
    fig_dir="./correlation_strength_values_Aoi_4.pdf",
)

list_metrics = "Strength_Atc_0"
list_xlabels = "Connection strength of ${\mathcal{G}^{(0)}_{TC_{1}}}^{[i]}$"
list_ylabels = "Connection strength of ${\mathcal{G}^{(0)}_{TC_{2}}}^{[i]}$"
plot_correlation_scatters(
    df_strengths,
    list_metrics,
    list_xlabels,
    list_ylabels,
    figsize=(8, 4),
    fig_dir="./correlation_strength_values_Atc_0.pdf",
)

list_metrics = "Strength_Atc_4"
list_xlabels = "Connection strength of ${\mathcal{G}^{(4)}_{TC_{1}}}^{[i]}$"
list_ylabels = "Connection strength of ${\mathcal{G}^{(4)}_{TC_{2}}}^{[i]}$"
plot_correlation_scatters(
    df_strengths,
    list_metrics,
    list_xlabels,
    list_ylabels,
    figsize=(8, 4),
    fig_dir="./correlation_strength_values_Atc_4.pdf",
)


Plotting the strength distribution of ${\mathcal{G}^{(k=0,4)}_{II_{1,2}}}^{[n]}$, ${\mathcal{G}^{(k=0,4)}_{TC_{1,2}}}^{[n]}$ between males and females in each rs-fMRI recording.

In [ ]:
df_subjects_info_strength = pd.merge(
    df_subject_metadata, df_strengths, how="inner", on=["Subject"]
)

list_metrics = ["Strength_Aoi_0", "Strength_Aoi_4"]
list_ylabels = [
    r"Connection strength of ${\mathcal{G}^{(0)}_{OI_{1}}}^{[i]}$ and ${\mathcal{G}^{(0)}_{OI_{2}}}^{[i]}$",
    r"Connection strength of ${\mathcal{G}^{(4)}_{OI_{1}}}^{[i]}$ and ${\mathcal{G}^{(4)}_{OI_{2}}}^{[i]}$",
]
plot_metrics_distribution(
    df_subjects_info_strength,
    list_metrics,
    list_ylabels,
    figsize=(16, 5),
    fig_dir="./distribution_strength_values_Aoi.pdf",
)

list_metrics = ["Strength_Atc_0", "Strength_Atc_4"]
list_ylabels = [
    r"Connection strength of ${\mathcal{G}^{(0)}_{TC_{1}}}^{[i]}$ and ${\mathcal{G}^{(0)}_{TC_{2}}}^{[i]}$",
    r"Connection strength of ${\mathcal{G}^{(4)}_{TC_{1}}}^{[i]}$ and ${\mathcal{G}^{(4)}_{TC_{2}}}^{[i]}$",
]
plot_metrics_distribution(
    df_subjects_info_strength,
    list_metrics,
    list_ylabels,
    figsize=(16, 5),
    fig_dir="./distribution_strength_values_Atc.pdf",
)
